# 04 — Results summary

Parses the logs from `01`–`03` and compares them against the paper's Table 1 / 2 / 3 / 9 numbers. Writes a CSV to Drive and prints a markdown table you can paste into the BTP report.

In [ ]:
import os, re, csv
BASE = '/content/drive/MyDrive/R-GFM'
RESULTS_DIR = f'{BASE}/results'

PAPER = {
    'nc_1shot': {
        'wisconsin': 35.41, 'texas': 32.36, 'cornell': 36.71, 'citeseer': 57.54,
        'cora': 49.50, 'pubmed': 49.80, 'computers': 52.30, 'photo': 61.08,
    },
    'nc_3shot': {
        'wisconsin': 43.10, 'texas': 44.18, 'cornell': 45.39, 'citeseer': 73.98,
        'cora': 59.26, 'pubmed': 59.19, 'computers': 56.02, 'photo': 71.50,
    },
    'nc_5shot': {
        'wisconsin': 47.75, 'texas': 51.64, 'cornell': 47.97, 'citeseer': 74.59,
        'cora': 63.77, 'pubmed': 63.39, 'computers': 59.55, 'photo': 73.62,
    },
    'lp': {
        'wisconsin': 84.15, 'cornell': 85.90, 'citeseer': 90.88, 'pubmed': 88.62,
        'cora': 89.27, 'photo': 81.53, 'texas': 87.94,
    },
}

def parse_nc_log(path):
    if not os.path.exists(path):
        return None
    txt = open(path).read()
    m = re.findall(r'Test Accuracy:\s*([0-9.]+)\s*\+/-\s*([0-9.]+)', txt)
    return float(m[-1][0]) * 100 if m else None

def parse_lp_log(path):
    if not os.path.exists(path):
        return None
    txt = open(path).read()
    m = re.findall(r'\[Test\]\s*Acc\s*[0-9.]+\s*\|\s*AUC\s*([0-9.]+)', txt)
    return float(m[-1]) * 100 if m else None

rows = []
for ds, paper_val in PAPER['nc_1shot'].items():
    ours = parse_nc_log(f'{RESULTS_DIR}/nc_1shot/{ds}.log')
    rows.append({'task': 'NC-1shot', 'dataset': ds, 'ours': ours, 'paper': paper_val})
for ds, paper_val in PAPER['nc_3shot'].items():
    ours = parse_nc_log(f'{RESULTS_DIR}/nc_3shot/{ds}.log')
    rows.append({'task': 'NC-3shot', 'dataset': ds, 'ours': ours, 'paper': paper_val})
for ds, paper_val in PAPER['nc_5shot'].items():
    ours = parse_nc_log(f'{RESULTS_DIR}/nc_5shot/{ds}.log')
    rows.append({'task': 'NC-5shot', 'dataset': ds, 'ours': ours, 'paper': paper_val})
for ds, paper_val in PAPER['lp'].items():
    ours = parse_lp_log(f'{RESULTS_DIR}/lp/{ds}.log')
    rows.append({'task': 'LP-AUC', 'dataset': ds, 'ours': ours, 'paper': paper_val})

for r in rows:
    r['delta'] = round(r['ours'] - r['paper'], 2) if r['ours'] is not None else None

out_csv = f'{RESULTS_DIR}/rgfm_comparison.csv'
with open(out_csv, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['task', 'dataset', 'ours', 'paper', 'delta'])
    w.writeheader()
    for r in rows:
        w.writerow(r)
print('Wrote', out_csv)

print()
print('| Task | Dataset | Ours | Paper | Delta |')
print('|------|---------|-----:|------:|------:|')
for r in rows:
    ours = f"{r['ours']:.2f}" if r['ours'] is not None else 'n/a'
    delta = f"{r['delta']:+.2f}" if r['delta'] is not None else 'n/a'
    print(f"| {r['task']} | {r['dataset']} | {ours} | {r['paper']:.2f} | {delta} |")